Task: You will try to Improve the macro F-score by applying feature engineering, feature selection, using different ML models, or other methods.The winner individual will achieve an additional 5 bonus points.

#Description
In this challenge, you should create a classification model to classify records to one of the 11 types of heart/ECG rhythms in this dataset. These labels include normal heart condition and also conditions such as Atrial fibrillation or different types of Arrhythmias such as Sinus Tachycardia.


### 1. Environment setup and library imports

In [ ]:
!pip install catboost imbalanced-learn -q

import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from collections import Counter
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 26.7 MB/s eta 0:00:00


### 2. Check GPU availability

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU: NVIDIA A100-SXM4-80GB


### 3. Load data -  Dataset Description
This dataset contains ECG measurements extracted from 10,646 patients with 500 Hz sampling rate that features 11 common rhythms.

1.   train.csv - the training set
2.   test.csv - the test set
3.   sample_submission.csv - a sample submission file in the correct format

In [ ]:
train_df = pd.read_csv('Train.csv')
test_df = pd.read_csv('Test.csv')
sample_sub = pd.read_csv('sample-submission.csv')
# === FIX: Check actual column names ===
print(f"Sample submission columns: {sample_sub.columns.tolist()}")
id_col = sample_sub.columns[0]  # Use first column as ID

Sample submission columns: ['patient_ID', 'Rhythm']


### 4. Define advanced feature engineering function

In [ ]:
def advanced_features(df):
    df = df.copy()
    df['Gender'] = df['Gender'].map({'MALE': 1, 'FEMALE': 0}).fillna(0.5)
    eps = 1e-8

    # Rate features
    df['Rate_Ratio'] = df['AtrialRate'] / (df['VentricularRate'] + eps)
    df['Rate_Diff'] = df['AtrialRate'] - df['VentricularRate']
    df['Rate_Diff_Abs'] = abs(df['Rate_Diff'])
    df['Rate_Sum'] = df['AtrialRate'] + df['VentricularRate']
    df['Rate_Product'] = df['AtrialRate'] * df['VentricularRate'] / 10000
    df['Rate_Harmonic'] = 2*df['AtrialRate']*df['VentricularRate']/(df['AtrialRate']+df['VentricularRate']+eps)
    df['Rate_Geometric'] = np.sqrt(df['AtrialRate'] * df['VentricularRate'])
    df['Rate_CV'] = df[['AtrialRate','VentricularRate']].std(axis=1) / (df[['AtrialRate','VentricularRate']].mean(axis=1) + eps)

    # QT features
    df['QT_Ratio'] = df['QTCorrected'] / (df['QTInterval'] + eps)
    df['QT_Diff'] = df['QTCorrected'] - df['QTInterval']
    df['QRS_QT_Ratio'] = df['QRSDuration'] / (df['QTCorrected'] + eps)
    df['QRS_QTInt_Ratio'] = df['QRSDuration'] / (df['QTInterval'] + eps)
    df['QT_QRS_Diff'] = df['QTInterval'] - df['QRSDuration']
    df['RR_Interval'] = 60000 / (df['VentricularRate'] + eps)
    df['Bazett_Ratio'] = df['QTCorrected'] / (df['QTInterval'] / np.sqrt(df['RR_Interval']/1000) + eps)

    # Clinical thresholds
    df['Brady'] = (df['VentricularRate'] < 60).astype(int)
    df['Tachy'] = (df['VentricularRate'] > 100).astype(int)
    df['Severe_Tachy'] = (df['VentricularRate'] > 150).astype(int)
    df['Extreme_Tachy'] = (df['VentricularRate'] > 200).astype(int)
    df['Normal_HR'] = ((df['VentricularRate'] >= 60) & (df['VentricularRate'] <= 100)).astype(int)

    df['Wide_QRS'] = (df['QRSDuration'] > 120).astype(int)
    df['Very_Wide_QRS'] = (df['QRSDuration'] > 150).astype(int)
    df['Narrow_QRS'] = (df['QRSDuration'] <= 100).astype(int)
    df['Long_QT'] = (df['QTCorrected'] > 460).astype(int)

    # Atrial patterns
    df['Flutter_Range'] = ((df['AtrialRate'] >= 250) & (df['AtrialRate'] <= 350)).astype(int)
    df['AFIB_Range'] = (df['AtrialRate'] > 350).astype(int)
    df['High_Atrial'] = (df['AtrialRate'] > 150).astype(int)
    df['AV_Dissociation'] = (df['Rate_Diff_Abs'] > 100).astype(int)

    # Pattern detection
    df['AFIB_Pattern'] = ((df['AtrialRate'] > 300) & (df['Rate_Ratio'] > 1.5)).astype(int)
    df['Flutter_Pattern'] = ((df['AtrialRate'] >= 250) & (df['AtrialRate'] <= 350) & (df['Rate_Ratio'] >= 2)).astype(int)
    df['SVT_Pattern'] = ((df['VentricularRate'] > 150) & (df['QRSDuration'] <= 120) & (df['Rate_Ratio'] < 1.5)).astype(int)

    # Age features
    df['Age_Decade'] = df['PatientAge'] // 10
    df['Age_Sq'] = df['PatientAge'] ** 2 / 1000
    df['Age_Log'] = np.log1p(df['PatientAge'])
    df['Age_HR'] = df['PatientAge'] * df['VentricularRate'] / 1000
    df['Max_HR'] = 220 - df['PatientAge']
    df['HR_Reserve'] = df['Max_HR'] - df['VentricularRate']
    df['HR_Pct_Max'] = df['VentricularRate'] / (df['Max_HR'] + eps)

    # Polynomials
    for col in ['VentricularRate', 'AtrialRate', 'QRSDuration']:
        df[f'{col}_sq'] = df[col] ** 2 / 10000
        df[f'{col}_sqrt'] = np.sqrt(df[col])
        df[f'{col}_log'] = np.log1p(df[col])

    # Cross features
    df['VR_QRS'] = df['VentricularRate'] * df['QRSDuration'] / 10000
    df['AR_QRS'] = df['AtrialRate'] * df['QRSDuration'] / 10000
    df['Gender_Age'] = df['Gender'] * df['PatientAge']
    df['Gender_HR'] = df['Gender'] * df['VentricularRate']
    df['Age_QRS'] = df['PatientAge'] * df['QRSDuration'] / 1000

    # Row stats
    nc = ['VentricularRate', 'AtrialRate', 'QRSDuration', 'QTInterval', 'QTCorrected']
    df['Row_Mean'] = df[nc].mean(axis=1)
    df['Row_Std'] = df[nc].std(axis=1)
    df['Row_Range'] = df[nc].max(axis=1) - df[nc].min(axis=1)

    return df

### 5. Apply feature engineering and prepare X (features) and y (labels)


In [ ]:
train_df = advanced_features(train_df)
test_df = advanced_features(test_df)

le = LabelEncoder()
y = le.fit_transform(train_df['Rhythm'])
X = train_df.drop(['Rhythm', 'id', 'Id', 'ID'], axis=1, errors='ignore')
X_test = test_df.drop(['id', 'Id', 'ID'], axis=1, errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"\nFeatures: {X.shape[1]}, Classes: {len(np.unique(y))}")



Features: 69, Classes: 11


### 6. Define the models (GPU and baseline)
All classifiers used in training:

- `xgb_gpu` → XGBoost classifier
- `lgbm_gpu` → LightGBM classifier
- `catboost_gpu` → CatBoost classifier
- `logreg` → Logistic Regression as a simpler baseline model.

These models will all be trained in cross-validation and later blended together.


In [ ]:
# === GPU MODELS ===
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

models = {
    'cat1': CatBoostClassifier(
        iterations=3000, learning_rate=0.02, depth=8, l2_leaf_reg=3,
        loss_function='MultiClass', # No class_weights
        task_type='GPU', devices='0', random_state=42, verbose=0
    ),
    'cat2': CatBoostClassifier(
        iterations=2500, learning_rate=0.025, depth=10, l2_leaf_reg=5,
        loss_function='MultiClass', # No class_weights
        task_type='GPU', devices='0', random_state=123, verbose=0
    ),
    'cat3': CatBoostClassifier(
        iterations=2000, learning_rate=0.03, depth=6, l2_leaf_reg=1,
        loss_function='MultiClass', # No class_weights
        task_type='GPU', devices='0', random_state=456, verbose=0
    ),
    'lgb1': lgb.LGBMClassifier(
        n_estimators=2500, learning_rate=0.02, num_leaves=63, max_depth=10,
        # Removed class_weight='balanced'
        min_child_samples=5,
        reg_alpha=0.1, reg_lambda=1, device='gpu',
        random_state=42, verbose=-1
    ),
    'lgb2': lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.025, num_leaves=95, max_depth=12,
        # Removed class_weight='balanced'
        min_child_samples=3,
        reg_alpha=0.5, reg_lambda=2, device='gpu',
        random_state=123, verbose=-1
    ),
    'xgb1': xgb.XGBClassifier(
        n_estimators=2500, learning_rate=0.02, max_depth=8,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1,
        tree_method='hist', device='cuda', random_state=42
    ),
}



### 7. Initialize storage for OOF and test predictions

- `n_classes` = number of unique target classes.
- `oof[name]` = array of shape `(N_train, n_classes)` to store **out-of-fold (OOF)** predicted probabilities for each model.
- `test_preds[name]` = array of shape `(N_test, n_classes)` to store **averaged test predicted probabilities** across all folds.


In [ ]:
n_classes = len(np.unique(y))
oof = {n: np.zeros((len(X), n_classes)) for n in models}
test_preds = {n: np.zeros((len(X_test), n_classes)) for n in models}


### 8. Train models with Stratified K-Fold CV + SMOTE


In [ ]:
from imblearn.over_sampling import SMOTE

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n=== Fold {fold+1}/{N_SPLITS} ===")
    X_tr, y_tr = X.iloc[train_idx].copy(), y[train_idx].copy()
    X_val, y_val = X.iloc[val_idx], y[val_idx]

    # Aggressive SMOTE
    min_class_count = min(Counter(y_tr).values())
    k = min(3, min_class_count - 1) if min_class_count > 1 else 1

    try:
        smote = SMOTE(random_state=42, k_neighbors=k)
        X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
        print(f"  SMOTE: {len(X_tr)} -> {len(X_tr_res)} samples")
    except Exception as e:
        print(f"  SMOTE failed: {e}")
        X_tr_res, y_tr_res = X_tr, y_tr

    # No sample_weights calculation

    for name, model in models.items():
        import copy
        m = copy.deepcopy(model)

        try:
            # Removed sample_weight usage
            m.fit(X_tr_res, y_tr_res)

            oof[name][val_idx] = m.predict_proba(X_val)
            test_preds[name] += m.predict_proba(X_test) / N_SPLITS
            f1 = f1_score(y_val, np.argmax(oof[name][val_idx], axis=1), average='macro')
            print(f"  {name}: {f1:.5f}")
        except Exception as e:
            print(f"  {name} failed: {e}")



=== Fold 1/10 ===
  SMOTE: 6706 -> 26730 samples
  cat1: 0.72482
  cat2: 0.73306
  cat3: 0.72695
  lgb1: 0.64481
  lgb2: 0.65258
  xgb1: 0.61307

=== Fold 2/10 ===
  SMOTE: 6706 -> 26730 samples
  cat1: 0.57946
  cat2: 0.58162
  cat3: 0.57693
  lgb1: 0.58494
  lgb2: 0.58639
  xgb1: 0.58274

=== Fold 3/10 ===
  SMOTE: 6707 -> 26730 samples
  cat1: 0.61931
  cat2: 0.62082
  cat3: 0.62022
  lgb1: 0.61732
  lgb2: 0.61558
  xgb1: 0.61409

=== Fold 4/10 ===
  SMOTE: 6707 -> 26730 samples
  cat1: 0.60028
  cat2: 0.60090
  cat3: 0.59776
  lgb1: 0.59098
  lgb2: 0.53120
  xgb1: 0.59379

=== Fold 5/10 ===
  SMOTE: 6707 -> 26730 samples
  cat1: 0.61364
  cat2: 0.61824
  cat3: 0.61971
  lgb1: 0.61902
  lgb2: 0.62847
  xgb1: 0.61245

=== Fold 6/10 ===
  SMOTE: 6707 -> 26730 samples
  cat1: 0.62879
  cat2: 0.61360
  cat3: 0.60642
  lgb1: 0.61422
  lgb2: 0.61171
  xgb1: 0.61417

=== Fold 7/10 ===
  SMOTE: 6707 -> 26730 samples
  cat1: 0.68501
  cat2: 0.69065
  cat3: 0.68655
  lgb1: 0.71301
  lgb2: 0.

### 9. Evaluate each model using OOF macro-F1


In [ ]:
# OOF Scores
print("\n=== OOF Scores ===")
for n in models:
    if np.sum(oof[n]) > 0: # Check if model ran
        f1 = f1_score(y, np.argmax(oof[n], axis=1), average='macro')
        print(f"{n}: {f1:.5f}")



=== OOF Scores ===
cat1: 0.60393
cat2: 0.61039
cat3: 0.60304
lgb1: 0.60496
lgb2: 0.58580
xgb1: 0.56283


### 10. Optimize model blend and Create submission file



In [ ]:
# === OPTIMIZE BLEND ===
from scipy.optimize import minimize

def neg_f1(w):
    w = np.abs(w) / np.sum(np.abs(w))
    blend = sum(w[i] * oof[n] for i, n in enumerate(models) if np.sum(oof[n]) > 0)
    if isinstance(blend, int): return 0
    return -f1_score(y, np.argmax(blend, axis=1), average='macro')

valid_models = [n for n in models if np.sum(oof[n]) > 0]
if valid_models:
    res = minimize(neg_f1, [1]*len(valid_models), method='Nelder-Mead', options={'maxiter': 3000})
    w_opt = np.abs(res.x) / np.sum(np.abs(res.x))
    print(f"\nOptimal weights: {dict(zip(valid_models, w_opt.round(3)))}")

    final_oof = sum(w_opt[i] * oof[n] for i, n in enumerate(valid_models))
    final_test = sum(w_opt[i] * test_preds[n] for i, n in enumerate(valid_models))
    blend_f1 = f1_score(y, np.argmax(final_oof, axis=1), average='macro')
    print(f"\n*** FINAL BLENDED OOF Macro F1: {blend_f1:.5f} ***")

    # SUBMISSION
    final_pred = le.inverse_transform(np.argmax(final_test, axis=1))
    # Get ID from test_df
    id_cols = ['patient_ID', 'id', 'Id', 'ID']
    test_id_col = next((c for c in id_cols if c in test_df.columns), None)
    test_ids = test_df[test_id_col] if test_id_col else range(len(final_pred))

    submission = pd.DataFrame({test_id_col or 'patient_ID': test_ids, 'Rhythm': final_pred})
    submission.to_csv('submission.csv', index=False)
    print(f"Saved! Shape: {submission.shape}")
    print(submission['Rhythm'].value_counts())
else:
    print("No models trained successfully.")



Optimal weights: {'cat1': np.float64(0.167), 'cat2': np.float64(0.167), 'cat3': np.float64(0.167), 'lgb1': np.float64(0.167), 'lgb2': np.float64(0.167), 'xgb1': np.float64(0.167)}

*** FINAL BLENDED OOF Macro F1: 0.60679 ***
Saved! Shape: (3194, 2)
Rhythm
7     1186
1      566
8      547
9      475
10     208
5       98
0       86
2       26
3        1
6        1
Name: count, dtype: int64


## Summary Section
Added the following features:
1. Advanced feature engineering - to get a better structure
2. Stratified K-Fold + SMOTE - to keep class proportions consistent and to handle class imbalance without changing the validation distribution.
3. Multiple GPU-based models and Optimized model blending -  tree-based models (XGBoost, LightGBM, CatBoost) plus Logistic Regression as a simpler baseline.

 Also realized that Tree-based models dominate tabular data competitions because they naturally handle feature interactions and class imbalance better than neural networks, which need massive amounts of data to generalize.

## AI Usage Section

I used Gemini in Google colab to resolve errors and Claude for more version control or improvise. The main uses were:

1. **Code structuring and refactoring**  
   I asked the AI to help clean up my Colab notebook, separate dead code from active code, and organize the pipeline into clear steps
   - “Document this Colab notebook cell by cell so I know what each cell does and what the output is.”
   - “Explain how to structure the models dictionary and the OOF/test prediction arrays.”

2. **Methodological guidance**  
   I used the AI to confirm good practices for imbalanced classification and ensembling. This included switching from `class_weight` to using SMOTE and then blending OOF predictions with optimized weights.  
   Prompt:
   - “Show me how to do an OOF-based weighted blend of multiple models using scipy.optimize.”
3. **Hardware Selection Guidance**  
   Prompt:
   - “Show me which option is better for this run - in terms of time”

All final code was run, checked, and adjusted by me. The AI assisted in speeding up writing, documentation, and reminding me of standard patterns (SMOTE + CV + blending), but I was responsible for deciding which ideas to keep and for validating the results on the actual data.
